In [1]:
from sklearn.datasets import make_classification
import pandas as pd

X, y = make_classification(
    n_samples=1000,
    n_features=5,
    n_informative=3,
    n_redundant=1,
    class_sep=1.0,
    weights=[0.7, 0.3],
    random_state=42
)

df = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(X.shape[1])])
df["target"] = y

print(df.head())
print(f"\nDistribuição: {df['target'].value_counts().to_dict()}")

   feature_0  feature_1  feature_2  feature_3  feature_4  target
0  -0.038769  -0.649239  -0.224746  -1.346275   0.126879       0
1   1.005284  -1.373239   1.157346   0.126493   1.422799       0
2  -0.742455  -0.573257   1.688442  -2.588237   0.762562       0
3  -1.587158   1.758582  -0.930664   0.764614   2.415399       1
4  -0.941758   0.367913  -0.549360  -2.029919  -1.503957       0

Distribuição: {0: 699, 1: 301}


In [2]:
import numpy as np
import pandas as pd

def gerar_fraude(n_samples: int = 1000, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)

    fraude = rng.integers(0, 2, size=n_samples)

    valor_transacao = np.where(
        fraude,
        rng.uniform(500, 10000, n_samples),
        rng.uniform(10, 800, n_samples)
    ).round(2)

    hora_transacao = np.where(
        fraude,
        rng.integers(0, 6, n_samples),
        rng.integers(7, 23, n_samples)
    )

    distancia_ultima_compra = np.where(
        fraude,
        rng.uniform(100, 5000, n_samples),
        rng.uniform(0, 50, n_samples)
    ).round(1)

    tentativas_senha = np.where(
        fraude,
        rng.integers(2, 10, n_samples),
        rng.integers(1, 2, n_samples)
    )

    pais_diferente = np.where(
        fraude,
        rng.choice([0, 1], size=n_samples),
        rng.choice([0, 1], size=n_samples, p=[0.95, 0.05])
    )

    return pd.DataFrame({
        "valor_transacao":         valor_transacao,
        "hora_transacao":          hora_transacao,
        "distancia_ultima_compra": distancia_ultima_compra,
        "tentativas_senha":        tentativas_senha,
        "pais_diferente":          pais_diferente,
        "fraude":                  fraude
    })

df = gerar_fraude(n_samples=2000, seed=42)
print(df.groupby("fraude").mean().round(2))

        valor_transacao  hora_transacao  distancia_ultima_compra  \
fraude                                                             
0                399.59           14.41                    25.61   
1               5264.42            2.50                  2567.97   

        tentativas_senha  pais_diferente  
fraude                                    
0                   1.00            0.04  
1                   5.61            0.50  


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# 1. Separar features (X) e o target (y)
X = df.drop(columns=["fraude"]).values
y = df["fraude"].values

# 2. Dividir em dados de treino (80%) e teste (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Instanciar e treinar o modelo
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 4. Fazer as predições na base de teste e avaliar
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["Legítima", "Fraude"]))

              precision    recall  f1-score   support

    Legítima       1.00      1.00      1.00       200
      Fraude       1.00      1.00      1.00       200

    accuracy                           1.00       400
   macro avg       1.00      1.00      1.00       400
weighted avg       1.00      1.00      1.00       400



In [4]:
import joblib
import numpy as np
import os

joblib.dump(model, "model.pkl")
tamanho_kb = os.path.getsize("model.pkl") / 1024
print(f"Modelo salvo: model.pkl ({tamanho_kb:.1f} KB)")

model_carregado = joblib.load("model.pkl")
amostra = X_test[:5]

pred_original  = model.predict(amostra)
pred_carregado = model_carregado.predict(amostra)

assert np.array_equal(pred_original, pred_carregado), "Predições divergem!"
print("✅ Artefato validado")
print(f"Predições: {pred_carregado}")
print(f"Probabilidades: {model_carregado.predict_proba(amostra).round(3)}")

Modelo salvo: model.pkl (63.8 KB)
✅ Artefato validado
Predições: [0 0 1 0 0]
Probabilidades: [[1. 0.]
 [1. 0.]
 [0. 1.]
 [1. 0.]
 [1. 0.]]


In [ ]:
from huggingface_hub import HfApi, login

login(token="")

api = HfApi()

repo_url = api.create_repo(
    repo_id="YgorReis/fraud-detector-v1",
    repo_type="model",
    exist_ok=True,
    private=False
)
print(f"Repositório: {repo_url}")

Repositório: https://huggingface.co/YgorReis/fraud-detector-v1


In [3]:
from huggingface_hub import HfApi
import sklearn
import joblib as jl
import numpy as np

# Conecta na API (como você já logou antes, ele pega seu token da sessão)
api = HfApi()
repo_id = "YgorReis/fraud-detector-v1"

# 1. Cria o requirements.txt com as versões exatas que você usou no treino
requirements = (
    f"scikit-learn=={sklearn.__version__}\n"
    f"joblib=={jl.__version__}\n"
    f"numpy=={np.__version__}\n"
)

with open("requirements.txt", "w") as f:
    f.write(requirements)

# 2. Faz o upload dos 3 arquivos de uma vez
arquivos_para_subir = ["model.pkl", "README.md", "requirements.txt"]

for filename in arquivos_para_subir:
    api.upload_file(
        path_or_fileobj=filename,
        path_in_repo=filename,
        repo_id=repo_id,
        repo_type="model",
        commit_message=f"Upload automático: {filename}"
    )
    print(f"✅ {filename} publicado com sucesso!")

print(f"\n🚀 Tudo pronto! Acesse seu modelo aqui: https://huggingface.co/{repo_id}")

Processing Files (1 / 1): 100%|██████████| 65.3kB / 65.3kB,  205kB/s  
New Data Upload: 100%|██████████| 65.3kB / 65.3kB,  205kB/s  


✅ model.pkl publicado com sucesso!
✅ README.md publicado com sucesso!
✅ requirements.txt publicado com sucesso!

🚀 Tudo pronto! Acesse seu modelo aqui: https://huggingface.co/YgorReis/fraud-detector-v1


In [4]:
from huggingface_hub import hf_hub_download
import joblib
import time

def load_model(
    repo_id: str,
    filename: str = "model.pkl",
    force_download: bool = False
):
    t0 = time.time()
    
    # hf_hub_download é inteligente: se o arquivo já está no pc, ele não baixa de novo
    local_path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        force_download=force_download
    )
    
    model = joblib.load(local_path)
    
    elapsed = time.time() - t0
    origem = "Hugging Face (Forçado/Primeira vez)" if force_download else "Cache local"
    print(f"✅ Modelo carregado de {origem} em {elapsed:.3f}s")
    
    return model

# Usando o seu repositório real
REPO_ID = "YgorReis/fraud-detector-v1"

print("--- Primeira chamada ---")
model = load_model(REPO_ID)

print("\n--- Segunda chamada ---")
model = load_model(REPO_ID)

print("\n--- Terceira chamada (forçando o download) ---")
model = load_model(REPO_ID, force_download=True)

--- Primeira chamada ---
✅ Modelo carregado de Cache local em 1.307s

--- Segunda chamada ---
✅ Modelo carregado de Cache local em 0.103s

--- Terceira chamada (forçando o download) ---
✅ Modelo carregado de Hugging Face (Forçado/Primeira vez) em 0.264s


In [1]:
from fastapi import APIRouter
from pydantic import BaseModel, Field
import numpy as np

router = APIRouter()

# O seu repositório real
REPO_ID = "YgorReis/fraud-detector-v1"
_model = None

# Lazy loading: Só carrega o modelo quando a primeira requisição bater
def get_model():
    global _model
    if _model is None:
        # Puxa a função que você testou no 4.1
        from model_utils import load_model 
        _model = load_model(REPO_ID)
    return _model

# Schema Pydantic - A "catraca" da sua API. Só entra dado com o tipo certo.
class PredictInput(BaseModel):
    valor_transacao: float = Field(gt=0, description="Valor da transação em reais")
    hora_transacao: int = Field(ge=0, le=23, description="Hora do dia (0-23)")
    distancia_ultima_compra: float = Field(ge=0, description="Distância geográfica em km")
    tentativas_senha: int = Field(ge=1, description="Tentativas de senha antes da transação")
    pais_diferente: int = Field(ge=0, le=1, description="1 se país diferente, 0 se igual")

# Schema de Saída
class PredictOutput(BaseModel):
    prediction: int
    probability: float
    label: str
    model_version: str

@router.post("/predict", response_model=PredictOutput)
async def predict(input: PredictInput):
    model = get_model()

    # ORDEM CRÍTICA: Tem que bater com o treino
    features = np.array([[
        input.valor_transacao,
        input.hora_transacao,
        input.distancia_ultima_compra,
        input.tentativas_senha,
        input.pais_diferente
    ]])

    # Faz a predição
    prediction = int(model.predict(features)[0])
    # Pega a probabilidade da classe 1 (Fraude)
    probability = float(model.predict_proba(features)[0][1])
    label = "Fraude" if prediction == 1 else "Legítima"

    return PredictOutput(
        prediction=prediction,
        probability=round(probability, 4),
        label=label,
        model_version=REPO_ID
    )